# mT5-Gold Seq2Seq RAG

This notebook implements **Gold-context fine-tuning** for `google/mt5-base` and then uses the same saved generator in three RAG configurations:

1. **TF-IDF + mT5-Gold**
2. **BM25 + mT5-Gold**
3. **Dense + mT5-Gold**

## Experimental design

### Training
The model is trained exclusively on:

`processed_question + gold context -> processed_answer`

The gold context is built only from chunks that cover the annotated `source_pages`.

### Validation
The validation set is **not used to learn parameters**. It is used for:
- monitoring `eval_loss`,
- early stopping,
- selecting the best checkpoint,
- later end-to-end comparison of the three RAG configurations.

### Test
The test set is not used for tuning or model selection. Test inference is intentionally disabled by default in this notebook (`RUN_FINAL_TEST = False`) and is run only after the model and RAG configuration are finalized.

### Important notes
- The generator uses `processed_text` / `processed_question`, not lexical representation
- The model, tokenizer, configuration, training history, and generated predictions are saved

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

In [2]:
SEED = 42

MODEL_NAME = "google/mt5-base"

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

# Training gold context i RAG inference koriste isti maximanlni broj context chunks da smanje mismatch izmedju treninga i inferencije
RAG_TOP_K = 5
MAX_GOLD_CONTEXT_CHUNKS = RAG_TOP_K

LEARNING_RATE = 3e-5
NUM_TRAIN_EPOCHS = 5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 1

GENERATION_MAX_NEW_TOKENS = 128
GENERATION_NUM_BEAMS = 4
GENERATION_BATCH_SIZE = 4

TOP_K_EXPORTED = 10

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_BF16 = (
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)
USE_FP16 = torch.cuda.is_available() and not USE_BF16


In [3]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        "Pokreni notebook iz root direktorijuma projekta "
        "(direktorijuma koji sadrži folder 'data')."
    )

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data" / "splits" / "validation.jsonl"
TEST_PATH = PROJECT_ROOT / "data" / "splits" / "test.jsonl"

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "seq2seq"
    / "mt5_gold"
)

CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL_DIR = ARTIFACTS_DIR / "best_model"
PREPARED_DATA_DIR = ARTIFACTS_DIR / "prepared_data"

GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "generation"
    / "mt5_gold"
)

for directory in [
    ARTIFACTS_DIR,
    CHECKPOINT_DIR,
    BEST_MODEL_DIR,
    PREPARED_DATA_DIR,
    GENERATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

Device: cpu
Model: google/mt5-base


In [4]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def save_jsonl(records: list[dict], path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


for required_path in [
    CHUNKS_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


chunks = load_jsonl(CHUNKS_PATH)
train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Chunkovi: {len(chunks)}")
print(f"Train pitanja: {len(train_data)}")
print(f"Validation pitanja: {len(validation_data)}")
print(f"Test pitanja: {len(test_data)}")

# print(chunks[0])
# print(train_data[0])

Chunkovi: 356
Train pitanja: 100
Validation pitanja: 21
Test pitanja: 22


### Build gold context and training/validation examples

The gold context is selected only through `source_pages`.

If multiple chunks overlap the gold pages:
1. chunks with a larger number of overlapping gold pages have priority,
2. at most `MAX_GOLD_CONTEXT_CHUNKS` chunks are selected,
3. the selected chunks are then returned to the document's original order.

This ensures that retrieval scores are not used during gold training.

In [5]:
def chunk_pages(chunk: dict) -> set[int]:
    return set(
        range(
            int(chunk["pdf_page_start"]),
            int(chunk["pdf_page_end"]) + 1
        )
    )


def get_gold_chunks(
    source_pages,
    max_chunks: int = MAX_GOLD_CONTEXT_CHUNKS
) -> list[dict]:

    if isinstance(source_pages, int):
        source_pages = [source_pages]

    source_pages = set(source_pages)

    candidates = []

    for corpus_index, chunk in enumerate(chunks):
        overlap = len(
            chunk_pages(chunk) & source_pages
        )

        if overlap > 0:
            candidates.append(
                {
                    "overlap": overlap,
                    "corpus_index": corpus_index,
                    "chunk": chunk,
                }
            )

    candidates.sort(
        key=lambda item: (
            -item["overlap"],
            item["corpus_index"]
        )
    )

    selected = candidates[:max_chunks]

    selected.sort(
        key=lambda item: item["corpus_index"]
    )

    return [
        item["chunk"]
        for item in selected
    ]


def build_context_from_chunks(
    selected_chunks: list[dict]
) -> str:
    return "\n\n---\n\n".join(
        chunk["processed_text"].strip()
        for chunk in selected_chunks
        if chunk.get("processed_text", "").strip()
    )


def build_gold_context(example: dict) -> str:
    return build_context_from_chunks(
        get_gold_chunks(
            example["source_pages"]
        )
    )

In [6]:
TASK_PREFIX = (
    "Odgovori na pitanje na osnovu datog konteksta. "
    "Odgovor treba da bude tačan, sažet i podržan kontekstom."
)


def build_model_input(
    processed_question: str,
    context: str
) -> str:
    return (
        f"{TASK_PREFIX}\n"
        f"Pitanje: {processed_question}\n"
        f"Kontekst:\n{context}\n"
        "Odgovor:"
    )


def prepare_gold_examples(
    data: list[dict],
    split_name: str
) -> list[dict]:

    prepared = []
    missing_context_ids = []

    for example in data:
        context = build_gold_context(example)

        if not context.strip():
            missing_context_ids.append(
                example["id"]
            )
            continue

        target = example.get(
            "processed_answer",
            example.get("answer", "")
        ).strip()

        if not target:
            raise ValueError(
                f"Nedostaje target odgovor za ID "
                f"{example['id']} u splitu {split_name}."
            )

        prepared.append({
            "question_id": example["id"],
            "input_text": build_model_input(
                example["processed_question"],
                context
            ),
            "target_text": target,
            "source_pages": example["source_pages"],
            "n_gold_chunks": len(
                get_gold_chunks(
                    example["source_pages"]
                )
            ),
        })

    if missing_context_ids:
        raise ValueError(
            f"Gold context nedostaje za {split_name} IDs: "
            f"{missing_context_ids}. "
            "Proveri corpus/source_pages pre treninga."
        )

    print(f"{split_name} spremni primeri: {len(prepared)}")

    return prepared


train_gold = prepare_gold_examples(
    train_data,
    "train"
)

validation_gold = prepare_gold_examples(
    validation_data,
    "validation"
)

# Test se NAMERNO ne koristi za training niti izbor checkpoint-a

save_jsonl(
    train_gold,
    PREPARED_DATA_DIR / "train_gold.jsonl"
)

save_jsonl(
    validation_gold,
    PREPARED_DATA_DIR / "validation_gold.jsonl"
)

train spremni primeri: 100
validation spremni primeri: 21


### Load the tokenizer and inspect input lengths

Length statistics are saved to `input_length_stats.json`

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    legacy=True,
)


def sequence_lengths(
    records: list[dict],
    field: str
) -> list[int]:
    return [
        len(
            tokenizer(
                record[field],
                truncation=False,
                add_special_tokens=True
            )["input_ids"]
        )
        for record in records
    ]


train_input_lengths = sequence_lengths(
    train_gold,
    "input_text"
)

validation_input_lengths = sequence_lengths(
    validation_gold,
    "input_text"
)

train_target_lengths = sequence_lengths(
    train_gold,
    "target_text"
)

length_stats = {
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "train": {
        "n": len(train_input_lengths),
        "input_mean": float(
            np.mean(train_input_lengths)
        ),
        "input_median": float(
            np.median(train_input_lengths)
        ),
        "input_max": int(
            max(train_input_lengths)
        ),
        "input_over_limit": int(
            sum(
                length > MAX_INPUT_LENGTH
                for length in train_input_lengths
            )
        ),
        "target_max": int(
            max(train_target_lengths)
        ),
        "target_over_limit": int(
            sum(
                length > MAX_TARGET_LENGTH
                for length in train_target_lengths
            )
        ),
    },
    "validation": {
        "n": len(validation_input_lengths),
        "input_mean": float(
            np.mean(validation_input_lengths)
        ),
        "input_median": float(
            np.median(validation_input_lengths)
        ),
        "input_max": int(
            max(validation_input_lengths)
        ),
        "input_over_limit": int(
            sum(
                length > MAX_INPUT_LENGTH
                for length in validation_input_lengths
            )
        ),
    },
}

with (ARTIFACTS_DIR / "input_length_stats.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        length_stats,
        file,
        ensure_ascii=False,
        indent=2
    )


# print(json.dumps(length_stats, indent=2, ensure_ascii=False))


### Tokenize datasets

In [9]:
train_dataset = Dataset.from_list(
    train_gold
)

validation_dataset = Dataset.from_list(
    validation_gold
)


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=validation_dataset.column_names
)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

### Model and training configuration

The best checkpoint is selected based on **validation `eval_loss`**

In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

# Smanjuje memorijsku potrošnju tokom fine-tuninga.
model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    num_train_epochs=NUM_TRAIN_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    gradient_checkpointing=True,

    bf16=USE_BF16,
    fp16=USE_FP16,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    predict_with_generate=False,

    report_to="none",
    disable_tqdm=True,

    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]